# Sobol — sample generation

Generate the Sobol sequence of parameter sets used to evaluate the sensitivity
of SeapoPym to its biological parameters (Manuscript Section 2.4.3).

Run mode (test or production) and parameter bounds are read from
`parameters.yaml`. With `mode: test` the sequence contains `(2P+2)·N = 192`
samples (P=5, N=16). With `mode: production` it contains 1,190,700 samples.

Output: `data/sobol_samples.parquet`.

In [1]:
from pathlib import Path

import pandas as pd
import yaml
from SALib import ProblemSpec


def _project_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Project root marker {marker!r} not found.")


PROJECT_ROOT = _project_root()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

with open(PROJECT_ROOT / "parameters.yaml") as f:
    SOBOL = yaml.safe_load(f)["sobol"]

mode = SOBOL["mode"]
n_samples = SOBOL[mode]["n_samples_per_dim"]
param_names = SOBOL["parameters_ordered"]
param_bounds = [SOBOL["bounds"][p] for p in param_names]
print(f"Mode: {mode} | N = {n_samples:,} | total samples = {n_samples * (2*len(param_names)+2):,}")

Mode: test | N = 1,024 | total samples = 12,288


In [2]:
sp = ProblemSpec({
    "names": param_names,
    "groups": None,
    "bounds": param_bounds,
    "outputs": SOBOL["metrics"],
})
sp.sample_sobol(n_samples)

samples = pd.DataFrame(sp.samples, columns=param_names)
output = DATA_DIR / "sobol_samples.parquet"
samples.to_parquet(output)
print(f"Wrote {len(samples):,} samples to {output.name}")
samples.head()

Wrote 12,288 samples to sobol_samples.parquet


,energy_transfert,tr_0,gamma_tr,lambda_temperature_0,gamma_lambda_temperature
0,0.312977,301.014194,-0.469999,0.006789,0.062422
1,0.908654,301.014194,-0.469999,0.006789,0.062422
2,0.312977,390.710406,-0.469999,0.006789,0.062422
3,0.312977,301.014194,-0.667663,0.006789,0.062422
4,0.312977,301.014194,-0.469999,0.016929,0.062422
